# Auditoría de calidad de datos (tarea #12)

**Responsable:** Juan · Sprint 1

Revisa nulos, duplicados, fechas imposibles y duraciones de las etapas, y prueba si la
anonimización corrió las fechas (festivos, Carnaval, fin de año).

**Antes de correr:**
1. Conecta la VPN y avisa en el grupo.
2. `python -m hun.data.extraer auditoria` (baja solo conteos a `data/raw/`).
3. Desconecta la VPN. Este notebook ya no usa la base.

Reglas: no imprimas filas de pacientes; las salidas se borran solas con `nbstripout`.
Los resultados se pasan a mano a `docs/calidad_datos.md`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from hun import config
from hun.data import auditoria as au

RAW = config.RAW_DIR
leer = lambda nombre: pd.read_parquet(RAW / f"{nombre}.parquet")  # noqa: E731

TIPOS = {
    581479: "Rehabilitación",
    9203: "Urgencias",
    3045729: "Urgencia-tratamiento",
    9201: "Hospitalización",
}

## 1. Nulos por columna

In [ ]:
nulos = au.resumen_nulos(leer("auditoria_nulos"))
nulos

## 2. Duplicados

In [ ]:
filas = nulos.groupby("tabla")["filas"].first().to_dict()
duplicados = au.resumen_duplicados(leer("auditoria_duplicados"), filas)
duplicados

## 3. Etapas (`visit_detail`)

Huérfanas, sin fin, fin antes del inicio, etapas que empiezan antes que su visita y
duraciones de más de 30 días. `alertas` lista los problemas que pasan del 1 %.

In [ ]:
etapas = au.resumen_etapas(leer("auditoria_etapas"))
cols = [
    "visit_detail_concept_id",
    "visit_concept_id",
    "etapas",
    "alertas",
    "horas_p50",
    "horas_p90",
    "horas_p99",
]
cols += [c for c in etapas.columns if c.startswith("pct_")]
etapas[cols].round(2)

In [ ]:
largo = au.tramos_largos(leer("auditoria_etapas"))
largo["grupo"] = (
    largo["visit_detail_concept_id"].map(TIPOS).fillna(largo["visit_detail_concept_id"].astype(str))
    + " en "
    + largo["visit_concept_id"].map(TIPOS).fillna(largo["visit_concept_id"].astype(str))
)
tabla = largo.pivot_table(index="tramo", columns="grupo", values="pct", observed=False)
ax = tabla.plot(kind="bar", figsize=(11, 4))
ax.set_ylabel("% de etapas")
ax.set_title("Duración de las etapas")
plt.tight_layout()

## 4. Edades y sexo (por décadas)

In [ ]:
personas = leer("auditoria_personas")
print(
    "Visitas con edad imposible:",
    int(personas["edad_imposible"].sum()),
    f"({100 * personas['edad_imposible'].sum() / personas['visitas'].sum():.3f} %)",
)
personas.pivot_table(
    index="decada_edad", columns="gender_concept_id", values="visitas", aggfunc="sum"
)

## 5. ¿Las fechas están corridas por la anonimización?

Para cada servicio se mide el volumen de los festivos entre semana frente al mismo día de la
semana en las 3 semanas vecinas, corriendo los festivos de −21 a +21 días. Si las fechas son
reales, el hueco más hondo está en el desfase 0.

Rehabilitación es la prueba fuerte (citas programadas, cierra en festivos). En urgencias el
efecto festivo es más débil, así que su resultado es solo de apoyo.

In [ ]:
llegadas = leer("auditoria_llegadas_diarias")
resultados = {}
for tipo in [581479, 9203]:
    diario = au.serie_diaria(llegadas, tipo, "visit_concept_id", "visitas")
    festivos = au.festivos_entre_semana(diario.index.min(), diario.index.max())
    prueba = au.prueba_desplazamiento(diario, festivos)
    resultados[TIPOS[tipo]] = au.veredicto_desplazamiento(prueba)
    ax = prueba.plot(x="desfase", y="relativo_medio", marker="o", figsize=(9, 3), legend=False)
    ax.axvline(0, color="gray", ls="--")
    ax.set_title(f"{TIPOS[tipo]}: volumen relativo en festivo + desfase")
    ax.set_ylabel("volumen / semanas vecinas")
pd.DataFrame(resultados).T

### 5.1 La misma prueba en las etapas (por si `visit_detail` tiene otro corrimiento)

In [ ]:
etapas_diarias = leer("auditoria_etapas_diarias")
res_etapas = {}
for tipo in etapas_diarias["visit_detail_concept_id"].unique():
    diario = au.serie_diaria(etapas_diarias, tipo, "visit_detail_concept_id", "etapas")
    if diario.median() < 5:
        continue  # muy poco volumen para la prueba
    festivos = au.festivos_entre_semana(diario.index.min(), diario.index.max())
    res_etapas[TIPOS.get(tipo, tipo)] = au.veredicto_desplazamiento(
        au.prueba_desplazamiento(diario, festivos)
    )
pd.DataFrame(res_etapas).T

### 5.2 Carnaval y fin de año

Perfil del volumen relativo desde 7 días antes hasta 7 días después del inicio de cada evento.
Solo describe: lo que se espera ver se discute con el equipo antes de concluir.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for tipo in [581479, 9203]:
    diario = au.serie_diaria(llegadas, tipo, "visit_concept_id", "visitas")
    ini, fin = diario.index.min(), diario.index.max()
    carnaval = au.dias_carnaval(ini, fin)
    inicios_carnaval = carnaval[carnaval.to_series().diff().dt.days.fillna(99).gt(1).to_numpy()]
    anios = range(ini.year, fin.year + 1)
    eventos = {
        "Carnaval (sábado)": inicios_carnaval,
        "Navidad (24 dic)": [pd.Timestamp(a, 12, 24) for a in anios],
        "Año nuevo (31 dic)": [pd.Timestamp(a, 12, 31) for a in anios],
    }
    for ax, (nombre, fechas) in zip(axes, eventos.items(), strict=True):
        perfil = au.perfil_evento(diario, fechas)
        ax.plot(perfil["dia"], perfil["relativo_medio"], marker="o", label=TIPOS[tipo])
        ax.axvline(0, color="gray", ls="--")
        ax.set_title(nombre)
axes[0].set_ylabel("volumen / semanas vecinas")
axes[-1].legend()
plt.tight_layout()

## 6. Reparto por día de la semana

In [ ]:
pd.DataFrame(
    {
        TIPOS[t]: au.reparto_dia_semana(au.serie_diaria(llegadas, t, "visit_concept_id", "visitas"))
        for t in [581479, 9203, 3045729]
    }
).rename(index=dict(enumerate(["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]))).round(2)

## 7. Conclusiones

Pasa las cifras a `docs/calidad_datos.md` (solo agregados) y avisa al equipo del resultado de
la sección 5: de eso depende si Carnaval, festivos y diciembre sirven como factores externos.